In [ ]:
# Change this to your preferred framework (e.g., 'cuda', 'pytorch', 'triton', 'jax', 'mojo')
EVAL_LANG = 'cuda'

SAVE_GPU = True


<p>
  Implement a GPU program that performs Top-K Gating for Mixture of Experts (MoE) models. Given a logit matrix of shape <code>[M, E]</code> where M is the number of tokens and E is the number of experts, identify the k largest values in each row, extract their indices, and apply softmax to get mixing weights.
</p>

<p>
  For each row i, the operation computes:
  $$
  \begin{align}
  \text{indices}_i, \text{vals}_i &= \text{TopK}(\text{logits}_i, k) \\
  \text{vals}_i &= \text{logits}_i[\text{indices}_i] \\
  \text{weights}_i &= \text{Softmax}(\text{vals}_i)
  \end{align}
  $$
</p>

<p>
  The selected experts must remain ordered by descending logit value, matching the order returned by
  <code>topk</code>. The <code>topk_weights</code> array must correspond positionally to
  <code>topk_indices</code> in that same order.
</p>

<h2>Implementation Requirements</h2>
<ul>
  <li>External libraries are not permitted</li>
  <li>The <code>solve</code> function signature must remain unchanged</li>
  <li>The final result must be stored in the <code>topk_weights</code> and <code>topk_indices</code> arrays</li>
</ul>

<h2>Example 1:</h2>
<pre>
Input:
  logits = [[1.0, 2.0, 3.0, 4.0],
            [4.0, 3.0, 2.0, 1.0]]
  M = 2, E = 4, k = 2

Output:
  topk_weights = [[0.7311, 0.2689],
                  [0.7311, 0.2689]]
  topk_indices = [[3, 2],
                  [0, 1]]

Explanation:
Row 0: Top-2 values are 4.0 and 3.0 at indices 3 and 2.
       Softmax([4.0, 3.0]) = [0.7311, 0.2689]
Row 1: Top-2 values are 4.0 and 3.0 at indices 0 and 1.
       Softmax([4.0, 3.0]) = [0.7311, 0.2689]
</pre>

<h2>Constraints</h2>
<ul>
  <li>1 ≤ <code>M</code> ≤ 10,000 (number of tokens)</li>
  <li>1 ≤ <code>E</code> ≤ 256 (number of experts)</li>
  <li>1 ≤ <code>k</code> ≤ <code>E</code> (top-k selection, typically k=2)</li>
  <li>All tensors are stored on GPU</li>
  <li>Logits are 32-bit floats</li>
  <li>Indices are 32-bit integers</li>

  <li>Performance is measured with <code>M</code> = 1,024, <code>k</code> = 2</li>
</ul>


# CUDA

In [ ]:
%%writefile solution.cu
#include <cuda_runtime.h>

// logits, topk_weights, topk_indices are device pointers
extern "C" void solve(const float* logits, float* topk_weights, int* topk_indices, int M, int E,
                      int k) {}


# CUTE

In [ ]:
%%writefile solution.py
import cutlass
import cutlass.cute as cute


# logits, topk_weights, topk_indices are tensors on the GPU
@cute.jit
def solve(
    logits: cute.Tensor,
    topk_weights: cute.Tensor,
    topk_indices: cute.Tensor,
    M: cute.Int32,
    E: cute.Int32,
    k: cute.Int32,
):
    pass


# JAX

In [ ]:
%%writefile solution.py
import jax
import jax.numpy as jnp


# logits is a tensor on the GPU
@jax.jit
def solve(logits: jax.Array, M: int, E: int, k: int) -> tuple[jax.Array, jax.Array]:
    pass


# MOJO

In [ ]:
%%writefile solution.mojo
from std.gpu.host import DeviceContext
from std.gpu import block_dim, block_idx, thread_idx
from std.memory import UnsafePointer
from std.math import ceildiv


@export
def solve(
    logits: UnsafePointer[Float32, MutExternalOrigin],
    topk_weights: UnsafePointer[Float32, MutExternalOrigin],
    topk_indices: UnsafePointer[Int32, MutExternalOrigin],
    M: Int32,
    E: Int32,
    k: Int32,
) raises:
    pass


# Torch

In [ ]:
%%writefile solution.py
import torch


# logits, topk_weights, topk_indices are tensors on the GPU
def solve(
    logits: torch.Tensor,
    topk_weights: torch.Tensor,
    topk_indices: torch.Tensor,
    M: int,
    E: int,
    k: int,
):
    pass


# Triton

In [ ]:
%%writefile solution.py
import torch
import triton
import triton.language as tl


# logits, topk_weights, topk_indices are tensors on the GPU
def solve(
    logits: torch.Tensor,
    topk_weights: torch.Tensor,
    topk_indices: torch.Tensor,
    M: int,
    E: int,
    k: int,
):
    pass


# Evaluate Setup

In [ ]:
# --- Core Challenge Base ---
from abc import ABC, abstractmethod
from typing import Any, Dict, List


class ChallengeBase(ABC):
    def __init__(self, name: str, atol: float, rtol: float, num_gpus: int, access_tier: str):
        self.name = name
        self.atol = atol
        self.rtol = rtol
        self.num_gpus = num_gpus
        self.access_tier = access_tier

    @abstractmethod
    def reference_impl(self, *args, **kwargs):
        """
        Reference solution implementation.
        """
        pass

    @abstractmethod
    def get_solve_signature(self) -> Dict[str, Any]:
        """
        Get the function signature for solution.

        Returns:
            Dictionary with argtypes and restype for ctypes
        """
        pass

    @abstractmethod
    def generate_example_test(self) -> List[Dict[str, Any]]:
        """
        Generate an example test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass

    @abstractmethod
    def generate_functional_test(self) -> List[Dict[str, Any]]:
        """
        Generate functional test cases for this problem.

        Returns:
            List of test case dictionaries
        """
        pass

    @abstractmethod
    def generate_performance_test(self) -> List[Dict[str, Any]]:
        """
        Generate a performance test case for this problem.

        Returns:
            Dictionary with test case parameters
        """
        pass


# --- Challenge Logic ---
import ctypes
from typing import Any, Dict, List

import torch


class Challenge(ChallengeBase):
    def __init__(self):
        super().__init__(
            name="MoE Top-K Gating", atol=1e-05, rtol=1e-05, num_gpus=1, access_tier="free"
        )

    def reference_impl(
        self,
        logits: torch.Tensor,
        topk_weights: torch.Tensor,
        topk_indices: torch.Tensor,
        M: int,
        E: int,
        k: int,
    ):
        """
        Computes the Top-K gating for Mixture of Experts.

        For each row in logits, select the k highest values, apply softmax to them,
        and return the weights and indices.
        """
        assert logits.shape == (M, E)
        assert topk_weights.shape == (M, k)
        assert topk_indices.shape == (M, k)
        assert logits.is_cuda and topk_weights.is_cuda and topk_indices.is_cuda
        assert topk_indices.dtype == torch.int32

        # 1. TopK Selection
        # logits: (M, E) -> vals: (M, k), indices: (M, k)
        vals, indices = torch.topk(logits, k, dim=-1)

        # 2. Softmax on the top k values
        weights = torch.softmax(vals, dim=-1)

        # 3. Write output
        topk_weights.copy_(weights)
        topk_indices.copy_(indices.to(torch.int32))

    def get_solve_signature(self) -> Dict[str, tuple]:
        return {
            "logits": (ctypes.POINTER(ctypes.c_float), "in"),
            "topk_weights": (ctypes.POINTER(ctypes.c_float), "out"),
            "topk_indices": (ctypes.POINTER(ctypes.c_int), "out"),
            "M": (ctypes.c_int, "in"),
            "E": (ctypes.c_int, "in"),
            "k": (ctypes.c_int, "in"),
        }

    def generate_example_test(self) -> Dict[str, Any]:
        dtype_float = torch.float32
        dtype_int = torch.int32
        M = 2
        E = 4
        k = 2

        # Example from problem description
        logits_data = torch.tensor(
            [[1.0, 2.0, 3.0, 4.0], [4.0, 3.0, 2.0, 1.0]], device="cuda", dtype=dtype_float
        )
        topk_weights_data = torch.zeros((M, k), device="cuda", dtype=dtype_float)
        topk_indices_data = torch.zeros((M, k), device="cuda", dtype=dtype_int)

        return {
            "logits": logits_data,
            "topk_weights": topk_weights_data,
            "topk_indices": topk_indices_data,
            "M": M,
            "E": E,
            "k": k,
        }

    def generate_functional_test(self) -> List[Dict[str, Any]]:
        dtype_float = torch.float32
        dtype_int = torch.int32
        test_cases = []

        # Test case 1: Basic example from problem description
        test_cases.append(
            {
                "logits": torch.tensor(
                    [[1.0, 2.0, 3.0, 4.0], [4.0, 3.0, 2.0, 1.0]], device="cuda", dtype=dtype_float
                ),
                "topk_weights": torch.zeros((2, 2), device="cuda", dtype=dtype_float),
                "topk_indices": torch.zeros((2, 2), device="cuda", dtype=dtype_int),
                "M": 2,
                "E": 4,
                "k": 2,
            }
        )

        # Test case 2: k=1 (single expert per token)
        test_cases.append(
            {
                "logits": torch.tensor(
                    [[5.0, 1.0, 3.0], [2.0, 8.0, 4.0], [6.0, 2.0, 9.0]],
                    device="cuda",
                    dtype=dtype_float,
                ),
                "topk_weights": torch.zeros((3, 1), device="cuda", dtype=dtype_float),
                "topk_indices": torch.zeros((3, 1), device="cuda", dtype=dtype_int),
                "M": 3,
                "E": 3,
                "k": 1,
            }
        )

        # Test case 3: k=E (all experts)
        test_cases.append(
            {
                "logits": torch.tensor(
                    [[1.0, 2.0, 3.0], [3.0, 1.0, 2.0]], device="cuda", dtype=dtype_float
                ),
                "topk_weights": torch.zeros((2, 3), device="cuda", dtype=dtype_float),
                "topk_indices": torch.zeros((2, 3), device="cuda", dtype=dtype_int),
                "M": 2,
                "E": 3,
                "k": 3,
            }
        )

        # Test case 4: Typical MoE configuration (M=4, E=8, k=2)
        torch.manual_seed(42)
        test_cases.append(
            {
                "logits": torch.randn((4, 8), device="cuda", dtype=dtype_float),
                "topk_weights": torch.zeros((4, 2), device="cuda", dtype=dtype_float),
                "topk_indices": torch.zeros((4, 2), device="cuda", dtype=dtype_int),
                "M": 4,
                "E": 8,
                "k": 2,
            }
        )

        # Test case 5: Larger E with small k (M=8, E=64, k=2)
        torch.manual_seed(123)
        test_cases.append(
            {
                "logits": torch.randn((8, 64), device="cuda", dtype=dtype_float),
                "topk_weights": torch.zeros((8, 2), device="cuda", dtype=dtype_float),
                "topk_indices": torch.zeros((8, 2), device="cuda", dtype=dtype_int),
                "M": 8,
                "E": 64,
                "k": 2,
            }
        )

        # Test case 6: Test with negative logits
        test_cases.append(
            {
                "logits": torch.tensor(
                    [[-1.0, -2.0, -3.0, -4.0], [-4.0, -1.0, -2.0, -3.0]],
                    device="cuda",
                    dtype=dtype_float,
                ),
                "topk_weights": torch.zeros((2, 2), device="cuda", dtype=dtype_float),
                "topk_indices": torch.zeros((2, 2), device="cuda", dtype=dtype_int),
                "M": 2,
                "E": 4,
                "k": 2,
            }
        )

        # Test case 7: Medium size test (M=100, E=16, k=4)
        torch.manual_seed(456)
        test_cases.append(
            {
                "logits": torch.randn((100, 16), device="cuda", dtype=dtype_float),
                "topk_weights": torch.zeros((100, 4), device="cuda", dtype=dtype_float),
                "topk_indices": torch.zeros((100, 4), device="cuda", dtype=dtype_int),
                "M": 100,
                "E": 16,
                "k": 4,
            }
        )

        return test_cases

    def generate_performance_test(self) -> Dict[str, Any]:
        dtype_float = torch.float32
        dtype_int = torch.int32
        M = 1024
        E = 64
        k = 2

        torch.manual_seed(789)
        return {
            "logits": torch.randn((M, E), device="cuda", dtype=dtype_float),
            "topk_weights": torch.zeros((M, k), device="cuda", dtype=dtype_float),
            "topk_indices": torch.zeros((M, k), device="cuda", dtype=dtype_int),
            "M": M,
            "E": E,
            "k": k,
        }


ch = Challenge()



In [ ]:
import os
import time
import ctypes
import torch

class Evaluate:
    @staticmethod
    def eval_cuda(ch):
        # 1. Compile a fresh uniquely named library
        so_filename = f'solution_func_{int(time.time())}.so'
        os.system(f'nvcc -shared -Xcompiler -fPIC -O3 solution.cu -o {so_filename}')
        lib = ctypes.CDLL(f'./{so_filename}')
        
        # 2. Extract signature and set argtypes
        signature = ch.get_solve_signature()
        lib.solve.argtypes = [arg_info[0] for arg_info in signature.values()]
        
        Evaluate._run_tests(ch, signature, lambda kwargs: lib.solve(*Evaluate._build_cuda_args(kwargs, signature)))

    @staticmethod
    def eval_python(ch):
        import importlib.util
        import sys
        
        spec = importlib.util.spec_from_file_location("solution", "solution.py")
        solution = importlib.util.module_from_spec(spec)
        sys.modules["solution"] = solution
        spec.loader.exec_module(solution)
        
        signature = ch.get_solve_signature()
        Evaluate._run_tests(ch, signature, lambda kwargs: Evaluate._run_python(solution, kwargs))

    @staticmethod
    def _run_python(solution, kwargs):
        solution.solve(**kwargs)
        if torch.cuda.is_available():
            torch.cuda.synchronize()

    @staticmethod
    def eval_mojo(ch):
        print("Mojo evaluation is currently executed via a separate runner or wrapper.")
        print("Ensure you have the mojo compiler installed and use 'mojo build solution.mojo' + ctypes/ffi,")
        print("or run an external python bridge. This is a stub.")

    @staticmethod
    def _build_cuda_args(kwargs, signature):
        cuda_args = []
        for k, (arg_type, dir_type) in signature.items():
            val = kwargs[k]
            if isinstance(val, torch.Tensor):
                cuda_args.append(ctypes.cast(val.data_ptr(), arg_type))
            else:
                cuda_args.append(arg_type(val))
        return cuda_args

    @staticmethod
    def _run_tests(ch, signature, run_fn):
        print("=== Running Functional Tests ===")
        functional_tests = ch.generate_functional_test()
        all_passed = True
        
        for i, test in enumerate(functional_tests):
            ref_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            test_kwargs = {k: (v.clone() if isinstance(v, torch.Tensor) else v) for k, v in test.items()}
            
            # Run Reference
            ch.reference_impl(**ref_kwargs)
            
            # Run implementation
            run_fn(test_kwargs)
            if torch.cuda.is_available():
                torch.cuda.synchronize()
            
            # Verify outputs
            match = True
            for k, (_, dir_type) in signature.items():
                if dir_type == "out":
                    if not torch.allclose(ref_kwargs[k], test_kwargs[k], atol=ch.atol, rtol=ch.rtol):
                        match = False
                        print(f"❌ Test {i+1}/{len(functional_tests)} Failed on output '{k}'")
                        break
            
            if match:
                print(f"✅ Test {i+1}/{len(functional_tests)} Passed")
            else:
                all_passed = False
                break
                
        if all_passed:
            print("\n🎉 All functional tests passed!")
            return True
        else:
            return False



# Evaluation code

In [ ]:
# Run the evaluator based on configuration
if EVAL_LANG == 'cuda':
    Evaluate.eval_cuda(ch)
elif EVAL_LANG in ['pytorch', 'triton', 'jax', 'cute']:
    Evaluate.eval_python(ch)
elif EVAL_LANG == 'mojo':
    Evaluate.eval_mojo(ch)
else:
    print(f"Unknown language {EVAL_LANG}")

# Disconnect runtime to save Colab resources
if SAVE_GPU:
    from google.colab import runtime
    runtime.unassign()
